# Football3D — HMR2 poses for clip04 (our own clip)

Pose four players across the whole clip: frames 0–336 (6.7 s at 49.95 fps), tracks **10, 15, 17, 9**.
Track **10 is the shooter** — it is 2.1 m from the ball at frame 192 and 2.2 m at 196, the nearest
player on both sides of contact (the kick is inside the detector's 193–195 gap, because a ball being
struck is motion-blurred). The other three sit 13–18 m from the goal, in the ball's path.

**Why clip04 is a better pose target than SNGS-043**, measured: boxes are **86–98 px** tall here
(median 92 across the clip) against the **78 px** of SNGS-043's shooter, and these tracks are
**unbroken** — 313–337 boxes out of 337 frames, no joined fragments. SNGS-043 averaged 5.5 fragments
per real player, and track 171's worst facing error came from a join hand-over. That failure mode
does not exist here, so no `TARGETS` entry needs several raw ids.

**What clip04 does NOT have: any ground truth.** No PA-MPJPE, no labelled facing. The only check
available is the direction-of-travel proxy in `src/pose/orientation_error.py`, whose own floor is
about 10°. Treat the numbers accordingly.

**Attach:** the clip04 frames dataset (`jazzeljs/clip04-data-football`), `clip04_pose_inputs.zip`
from `src/pose/export_kaggle_inputs.py`, your private SMPL input, and — if you saved one — the
private `hmr2_cache` dataset. Enable Internet and a **T4 GPU**.

**HMR2 cache (2.7 GB):** the first run downloads and extracts it to `/kaggle/working/hmr2_cache`.
Save that output as a **private** dataset and attach it next time; cell 3 then skips the download.
The cache never contains your SMPL file.

**Cost:** 4 tracks x ~330 frames = about 1300 crops, a few times the SNGS-043 run. Fits one session.


In [ ]:
from pathlib import Path
import json, numpy as np

INPUT = Path('/kaggle/input')
# clip04's frames are 00000.jpg (5 digits, from 0), NOT SoccerNet's 000001.jpg (6 digits, from 1).
frame = next(iter(INPUT.rglob('00000.jpg')), None)
TRACK_JSON = next(iter(INPUT.rglob('clip04_football-player-detection-v9_botsort.json')), None)
SMPL_SOURCE = next(iter(INPUT.rglob('basicmodel_m_lbs_10_207_0_v1.1.0.pkl')), None)
MANIFEST = next(iter(INPUT.rglob('manifest.json')), None)
assert frame and TRACK_JSON and SMPL_SOURCE, 'Attach clip04 frames + clip04_pose_inputs.zip + private SMPL input'
FRAMES = frame.parent

TARGETS = {10: [10], 15: [15], 17: [17], 9: [9]}  # id 10 is the shooter; no joins needed on clip04
START, END = 0, 336        # the whole clip
FPS = 49.95                # clip04's rate. SoccerNet is 25 -- do not copy that constant here.

# The manifest is the guard against a stale upload. export_kaggle_inputs.py recorded how many
# boxes each track HAS on the laptop; if Kaggle disagrees, the attached JSON is the wrong one and
# we would happily pose a different player for an hour.
if MANIFEST:
    manifest = json.loads(MANIFEST.read_text())
    assert manifest['clip'] == 'clip04', manifest['clip']
    assert abs(manifest['fps'] - FPS) < 0.01, manifest['fps']
    print('manifest boxes per track:', manifest['boxes_per_track'])
    print('manifest median box height px:', manifest['median_box_height_px'])
WORK = Path('/kaggle/working/football3d_clip04_poses')
WORK.mkdir(parents=True, exist_ok=True)
print('targets:', TARGETS, '| frames:', START, 'to', END, '| inputs:', len(list(FRAMES.glob('*.jpg'))))


In [ ]:
import shutil, subprocess, sys

REPO = Path('/kaggle/working/4D-Humans')
if not (REPO / 'hmr2' / '__init__.py').is_file():
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/shubham-goel/4D-Humans.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for source in REPO.rglob('*.py'):
    text = source.read_text()
    if 'timm.models.layers' in text: source.write_text(text.replace('timm.models.layers', 'timm.layers'))
if not shutil.which('aria2c'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)
sys.path.insert(0, str(REPO))

In [ ]:
import os, tarfile
from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import DEFAULT_CHECKPOINT

CACHE, checkpoint = Path(CACHE_DIR_4DHUMANS), Path(DEFAULT_CHECKPOINT)
relative = checkpoint.relative_to(CACHE).as_posix()
SAVE_CACHE = Path('/kaggle/working/hmr2_cache')
def cache_root(folder):
    found = next((p for p in folder.rglob(checkpoint.name) if p.as_posix().endswith(relative)), None)
    return Path(found.as_posix()[:-len(relative)]) if found else None

# Reuse an attached private cache, or recover the earlier buggy extraction in WORK.
source = cache_root(INPUT) or cache_root(SAVE_CACHE) or cache_root(WORK)
if source is not None and not checkpoint.exists():
    shutil.copytree(source, CACHE, dirs_exist_ok=True, copy_function=os.symlink)
if not checkpoint.exists():
    SAVE_CACHE.mkdir(parents=True, exist_ok=True)
    archive = SAVE_CACHE / 'hmr2_data.tar.gz'
    subprocess.run(['aria2c', '--continue=true', '-x', '16', '-s', '16', '-k', '1M', '--file-allocation=none', '-d', str(SAVE_CACHE), '-o', archive.name, 'https://www.cs.utexas.edu/~pavlakos/4dhumans/hmr2_data.tar.gz'], check=True)
    assert not Path(str(archive) + '.aria2').exists(), 'Download incomplete: rerun this cell'
    with tarfile.open(archive, 'r:*') as bundle: bundle.extractall(SAVE_CACHE, filter='data')
    archive.unlink()
    source = cache_root(SAVE_CACHE)
    assert source is not None, f'checkpoint not found under {SAVE_CACHE}'
    shutil.copytree(source, CACHE, dirs_exist_ok=True, copy_function=os.symlink)
assert checkpoint.exists(), f'missing checkpoint: {checkpoint}'
for folder in [CACHE, *CACHE.rglob('*')]:
    if folder.is_dir() and not folder.is_symlink(): folder.chmod(0o755)
cache_model = CACHE / 'data/smpl/SMPL_NEUTRAL.pkl'
cache_model.parent.mkdir(parents=True, exist_ok=True)
cache_model.unlink(missing_ok=True)  # never save the licensed SMPL file in hmr2_cache
shutil.copy2(SMPL_SOURCE, cache_model)
print('checkpoint:', checkpoint)

In [ ]:
import contextlib, io
import cv2, torch
from hmr2.models import load_hmr2
from hmr2.datasets.vitdet_dataset import ViTDetDataset
from hmr2.utils import recursive_to

assert torch.cuda.is_available(), 'Enable a T4 GPU in Kaggle settings'
track = json.loads(TRACK_JSON.read_text())
# Ball boxes share the id counter with players: on clip04 ids 21 and 24 are used by BOTH a
# ball box and a player box, so an unfiltered lookup can hand a ball to a pose model. None
# of 9/10/15/17 collide, but filter anyway -- the next person picks different ids.
boxes = {f['frame']: {b['id']: b['xyxy'] for b in f['boxes'] if b['cls'] != 'ball'}
         for f in track['frames']}
counts = {target: sum(any(raw_id in boxes.get(frame, {}) for raw_id in raw_ids) for frame in range(START, END + 1)) for target, raw_ids in TARGETS.items()}
print('tracked boxes in selected window:', counts)
assert all(counts[target] >= 100 for target in TARGETS), 'Wrong or stale raw track JSON attached to Kaggle; update that dataset before inference'
if MANIFEST:
    expected = {int(k): v for k, v in manifest['boxes_per_track'].items()}
    assert counts == expected, f'Kaggle sees {counts}, the laptop packed {expected} -- stale dataset'
original_load = torch.load
def trusted_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = trusted_load
try: model, config = load_hmr2(str(checkpoint))
finally: torch.load = original_load
model = model.cuda().eval()

for target, raw_ids in TARGETS.items():
    saved = {key: [] for key in ['frame', 'joints', 'global_orient_rotmat', 'body_pose_rotmat', 'betas', 'pred_cam', 'pred_cam_t']}
    for frame in range(START, END + 1):
        box = next((boxes.get(frame, {}).get(raw_id) for raw_id in raw_ids if raw_id in boxes.get(frame, {})), None)
        if box is None: continue
        image = cv2.imread(str(FRAMES / f'{frame:05d}.jpg'))  # 5 digits: clip04, not SoccerNet
        # HMR2 prints one downsampling_factor line per crop; it is normal but unhelpful.
        with contextlib.redirect_stdout(io.StringIO()):
            dataset = ViTDetDataset(config, image, np.asarray(box, dtype=np.float32)[None])
        batch = recursive_to(next(iter(torch.utils.data.DataLoader(dataset, batch_size=1))), 'cuda')
        with torch.inference_mode(): result = model(batch)
        params = result['pred_smpl_params']
        male = model.smpl(global_orient=params['global_orient'].float(), body_pose=params['body_pose'].float(), betas=torch.zeros_like(params['betas']).float(), pose2rot=False)
        joints = torch.einsum('jk,kv->jv', model.smpl.J_regressor.cuda(), male.vertices[0])
        saved['frame'].append(frame); saved['joints'].append(joints.cpu().numpy())
        saved['global_orient_rotmat'].append(params['global_orient'][0].cpu().numpy())
        saved['body_pose_rotmat'].append(params['body_pose'][0].cpu().numpy())
        saved['betas'].append(params['betas'][0].cpu().numpy())
        saved['pred_cam'].append(result['pred_cam'][0].cpu().numpy())
        saved['pred_cam_t'].append(result['pred_cam_t'][0].cpu().numpy())
        if len(saved['frame']) % 25 == 0: print(f'track {target}: {len(saved["frame"])} frames')
    out = WORK / f'pose_{target}.npz'
    np.savez_compressed(out, **{key: np.asarray(value) for key, value in saved.items()}, fps=np.float32(FPS))
    print('saved', out.name, 'frames:', len(saved['frame']))

In [ ]:
from IPython.display import FileLink, display
for target in TARGETS: display(FileLink(str(WORK / f'pose_{target}.npz')))